# 003 · File System Operations with os and shutil

> **Standard Library › Files & Paths** · Python 3.14

`os` exposes the operating system's low-level file operations (list, stat, rename, delete), and `shutil` ("shell utilities") adds the high-level ones: copying with metadata, copying or deleting whole trees, moving across drives, archives and finding executables.

**In this notebook you will learn:**

- listing directories with `os.listdir`, `os.scandir` and `os.walk`
- creating, renaming, replacing and deleting with `os`
- recognising the legacy `os.path` string API (and its `pathlib` replacement)
- reading metadata with `os.stat`
- `shutil.copy`/`copy2`/`copytree`/`move`/`rmtree`, archives, `disk_usage` and `which`

## .NET mental model

| .NET | Python |
|---|---|
| `Directory.GetFileSystemEntries` | `os.listdir` / `os.scandir` |
| `Directory.CreateDirectory` | `os.makedirs(p, exist_ok=True)` |
| `File.Delete` / `Directory.Delete(p)` | `os.remove` / `os.rmdir` |
| `Directory.Delete(p, recursive: true)` | `shutil.rmtree(p)` |
| `File.Move(a, b, overwrite: true)` | `os.replace(a, b)` |
| `File.Copy(a, b, overwrite: true)` | `shutil.copy` / `shutil.copy2` |
| (no built-in recursive directory copy) | `shutil.copytree` |
| `System.IO.Compression.ZipFile.CreateFromDirectory` | `shutil.make_archive` |
| `DriveInfo.TotalSize` / `AvailableFreeSpace` | `shutil.disk_usage` |
| `FileInfo.Length`, `LastWriteTime` | `os.stat(p).st_size`, `.st_mtime` |

**Tip:** since Python 3.4 the recommended way to *represent* paths is `pathlib.Path` *(see Standard Library › Files & Paths › 001 · Paths with pathlib)*. Every `os` and `shutil` function accepts `Path` objects, so the usual modern style is: `Path` for paths, `shutil` for the heavy lifting.

We'll work in a sandbox and build a small project tree. `rel()` prints paths relative to it so the output is machine-independent.

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
root = Path(tmp.name)

def rel(p) -> str:
    return Path(p).relative_to(root).as_posix()

for name in ["app/main.py", "app/util.py", "app/data/orders.csv", "app/__pycache__/main.cpython-314.pyc"]:
    f = root / name
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(f"# {f.name}\n", encoding="utf-8")

print(sorted(rel(p) for p in root.rglob("*")))

## Listing directories

- `os.listdir(p)` — just the **names** (strings), in arbitrary order.
- `os.scandir(p)` — an iterator of `DirEntry` objects that already know whether they're a file or directory, without extra system calls. Use it as a context manager. It's the fast choice for big directories.

```csharp
// C#
foreach (var entry in new DirectoryInfo("app").EnumerateFileSystemInfos())
    Console.WriteLine($"{entry.Name} {entry is DirectoryInfo}");
```

In [ ]:
app = root / "app"
print(sorted(os.listdir(app)))

with os.scandir(app) as entries:
    for entry in sorted(entries, key=lambda e: e.name):
        kind = "dir " if entry.is_dir() else "file"
        print(kind, entry.name, entry.stat().st_size if entry.is_file() else "")

### `os.walk`

`os.walk(top)` yields `(dirpath, dirnames, filenames)` with **strings** for every directory under `top` — the classic way to traverse a tree. Edit `dirnames` in place to skip directories. (`Path.walk()` is the `pathlib` version, added in 3.12.)

In [ ]:
for dirpath, dirnames, filenames in os.walk(app):
    dirnames[:] = sorted(d for d in dirnames if d != "__pycache__")   # prune + order
    print(rel(dirpath), dirnames, sorted(filenames))

## Creating, renaming and deleting

| Operation | Python | .NET |
|---|---|---|
| make one dir | `os.mkdir(p)` | `Directory.CreateDirectory` (always creates parents) |
| make dir + parents | `os.makedirs(p, exist_ok=True)` | `Directory.CreateDirectory` |
| rename / move | `os.rename(a, b)` | `File.Move(a, b)` |
| rename, overwrite target | `os.replace(a, b)` | `File.Move(a, b, overwrite: true)` |
| delete file | `os.remove(p)` (alias `os.unlink`) | `File.Delete` |
| delete **empty** dir | `os.rmdir(p)` | `Directory.Delete(p)` |

In [ ]:
os.makedirs(root / "out" / "2026" / "10", exist_ok=True)
(root / "out" / "draft.txt").write_text("v1", encoding="utf-8")
(root / "out" / "final.txt").write_text("old", encoding="utf-8")

os.replace(root / "out" / "draft.txt", root / "out" / "final.txt")   # overwrites
print(sorted(os.listdir(root / "out")), (root / "out" / "final.txt").read_text(encoding="utf-8"))

os.remove(root / "out" / "final.txt")
os.rmdir(root / "out" / "2026" / "10")         # only works on empty directories
print(sorted(os.listdir(root / "out")))

⚠️ **Gotcha:** `os.rename` overwrites an existing target **on POSIX but raises on Windows** — portable code should use `os.replace` when overwriting is intended. Also, `os.rmdir` refuses non-empty directories, and `os.remove` refuses directories (the exact `OSError` subclass varies by OS — `PermissionError` on macOS, `IsADirectoryError` on Linux):

In [ ]:
for action in (lambda: os.rmdir(app), lambda: os.remove(app)):
    try:
        action()
    except OSError as e:
        print(f"{type(e).__name__}: {e.strerror}")

## The legacy `os.path` module — recognise it

Lots of existing code (and Stack Overflow answers) use `os.path`, which works on plain strings — the counterpart of `System.IO.Path`. You should be able to read it, but prefer `pathlib` in new code.

| `os.path` | `pathlib` |
|---|---|
| `os.path.join(a, b)` | `Path(a) / b` |
| `os.path.basename(p)` / `dirname(p)` | `p.name` / `p.parent` |
| `os.path.splitext(p)` | `(p.stem, p.suffix)` (roughly) |
| `os.path.exists(p)` / `isfile` / `isdir` | `p.exists()` / `is_file()` / `is_dir()` |
| `os.path.abspath(p)` | `p.absolute()` (or `p.resolve()`) |
| `os.path.getsize(p)` | `p.stat().st_size` |
| `os.path.expanduser("~/x")` | `Path("~/x").expanduser()` |

In [ ]:
s = os.path.join("app", "data", "orders.csv")      # a plain str
print(s, type(s).__name__)
print(os.path.basename(s), os.path.dirname(s))
print(os.path.splitext(s))
print(os.path.isfile(root / s))                     # accepts Path objects too

## File metadata with `os.stat`

`os.stat(p)` (or `Path.stat()`) returns a `stat_result` — think `FileInfo` properties. Times are **Unix timestamps** (float seconds); convert with `datetime.fromtimestamp` *(see Standard Library › Dates & Times › 001 · Dates, Times and Timedeltas)*.

In [ ]:
from datetime import datetime, timezone

orders = app / "data" / "orders.csv"
fixed = datetime(2026, 1, 15, 9, 30, tzinfo=timezone.utc).timestamp()
os.utime(orders, (fixed, fixed))       # set access & modified time (File.SetLastWriteTimeUtc)

info = os.stat(orders)
print(info.st_size, "bytes")                                       # FileInfo.Length
print(datetime.fromtimestamp(info.st_mtime, tz=timezone.utc))      # LastWriteTimeUtc
print(oct(info.st_mode & 0o777))                                   # Unix permission bits (umask dependent)

## Copying files with `shutil`

| Function | Copies | Notes |
|---|---|---|
| `shutil.copyfile(src, dst)` | content only | `dst` must be a file path |
| `shutil.copy(src, dst)` | content + permission bits | `dst` may be a directory |
| `shutil.copy2(src, dst)` | content + permissions + **timestamps** | closest to Explorer/Finder copy |

`File.Copy` in .NET copies content and attributes; `copy2` is the closest match. All of them **overwrite** an existing destination silently.

In [ ]:
backup = root / "backup"
backup.mkdir()

c1 = shutil.copy(orders, backup / "orders-copy.csv")
c2 = shutil.copy2(orders, backup)                  # into the directory, keeps the name
print(rel(c1), rel(c2))                            # both return the destination path

def same_mtime(p) -> bool:
    return os.stat(p).st_mtime == os.stat(orders).st_mtime

print("copy keeps mtime: ", same_mtime(c1))        # False — gets "now"
print("copy2 keeps mtime:", same_mtime(c2))        # True

## Whole trees: `copytree` and `rmtree`

```csharp
// C#
Directory.Delete("backup", recursive: true);
// there is no Directory.Copy — you write a recursive loop yourself
```

`shutil.copytree(src, dst)` copies recursively (using `copy2` by default). `ignore=shutil.ignore_patterns(...)` skips matching names, and `dirs_exist_ok=True` lets you copy into an existing directory.

In [ ]:
dest = shutil.copytree(app, root / "release", ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
print(rel(dest))
print(sorted(rel(p) for p in dest.rglob("*")))

`shutil.rmtree` is `Directory.Delete(path, recursive: true)` — it deletes **everything** below the path with no confirmation and no recycle bin. Double-check the path you pass it!

In [ ]:
shutil.rmtree(root / "release")
print((root / "release").exists())    # False

try:
    shutil.rmtree(root / "release")   # already gone
except FileNotFoundError as e:
    print(f"{type(e).__name__}: {e.strerror}")

## Moving: `shutil.move`

`os.rename`/`os.replace` only work within one file system. `shutil.move(src, dst)` falls back to copy-then-delete when needed (e.g. a different drive), works for files **and** directories, and moves *into* `dst` if it is an existing directory.

In [ ]:
archive_dir = root / "archive"
archive_dir.mkdir()

moved = shutil.move(backup / "orders-copy.csv", archive_dir)   # into the folder
print(rel(moved))
moved_dir = shutil.move(backup, archive_dir / "old-backup")      # rename a whole directory
print(rel(moved_dir), sorted(os.listdir(moved_dir)))

## Archives: `make_archive` / `unpack_archive`

`shutil.make_archive(base_name, format, root_dir)` zips (or tars) a directory and returns the archive path; the extension is added for you. Formats: `"zip"`, `"tar"`, `"gztar"`, `"bztar"`, `"xztar"` (plus `"zstdtar"` when the 3.14 `compression.zstd` module is available). For individual archive entries use the `zipfile`/`tarfile` modules — the counterparts of `System.IO.Compression.ZipArchive`.

In [ ]:
zip_path = shutil.make_archive(root / "app-release", "zip", root_dir=app)
print(rel(zip_path))

shutil.unpack_archive(zip_path, root / "unpacked")
print(sorted(rel(p) for p in (root / "unpacked").rglob("*.py")))

## Disk usage and finding executables

- `shutil.disk_usage(p)` → named tuple `(total, used, free)` in bytes, like `DriveInfo`.
- `shutil.which("git")` → full path of an executable on `PATH`, or `None` — the cross-platform `where` / `which` (no .NET BCL equivalent).

The exact numbers and paths are machine-specific, so we only print derived values here.

In [ ]:
usage = shutil.disk_usage(root)
print(usage._fields)                                   # ('total', 'used', 'free')
print(usage.total >= usage.used and usage.free > 0)    # True

print(shutil.which("definitely-not-a-real-tool-xyz"))  # None
python_exe = shutil.which("python3") or shutil.which("python")
print(python_exe is not None)                           # True on most machines

In [ ]:
tmp.cleanup()
print(root.exists())

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| List names | `Directory.GetFileSystemEntries` | `os.listdir(p)` / `Path.iterdir()` |
| Fast listing with types | `EnumerateFileSystemInfos` | `os.scandir(p)` |
| Walk a tree | recursive `EnumerateFiles` | `os.walk(p)` / `Path.walk()` |
| Create dirs | `Directory.CreateDirectory` | `os.makedirs(p, exist_ok=True)` |
| Rename with overwrite | `File.Move(a, b, true)` | `os.replace(a, b)` |
| Delete file / empty dir | `File.Delete` / `Directory.Delete` | `os.remove` / `os.rmdir` |
| Delete tree | `Directory.Delete(p, true)` | `shutil.rmtree(p)` |
| Copy file (+metadata) | `File.Copy` | `shutil.copy2(a, b)` |
| Copy tree | manual recursion | `shutil.copytree(a, b)` |
| Move across drives | `File.Move` / `Directory.Move` (same volume) | `shutil.move(a, b)` |
| Zip a folder | `ZipFile.CreateFromDirectory` | `shutil.make_archive(base, "zip", dir)` |
| File size / times | `FileInfo.Length` / `LastWriteTime` | `os.stat(p).st_size` / `.st_mtime` |

## Exercises

1. In a temp dir, create `src/a.txt`, `src/b.log`, `src/sub/c.txt`. Copy `src` to `dst` with `shutil.copytree`, **excluding** `*.log` files, and print the sorted relative paths in `dst`.
2. Using `os.scandir`, write `largest_file(folder) -> str` that returns the name of the largest file directly inside `folder` (ignore subdirectories).
3. Zip the `dst` folder from exercise 1, delete `dst` with `shutil.rmtree`, then restore it with `shutil.unpack_archive` and confirm `sub/c.txt` exists again.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import os
import shutil
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    root = Path(d)

    # 1 — copytree with ignore_patterns
    for name, size in [("src/a.txt", 10), ("src/b.log", 5), ("src/sub/c.txt", 30)]:
        f = root / name
        f.parent.mkdir(parents=True, exist_ok=True)
        f.write_text("x" * size, encoding="utf-8")
    dst = shutil.copytree(root / "src", root / "dst", ignore=shutil.ignore_patterns("*.log"))
    print(sorted(p.relative_to(dst).as_posix() for p in dst.rglob("*")))

    # 2 — scandir, keep only files, max by size
    def largest_file(folder) -> str:
        with os.scandir(folder) as entries:
            files = [e for e in entries if e.is_file()]
        return max(files, key=lambda e: e.stat().st_size).name

    print(largest_file(root / "src"))   # a.txt (sub/c.txt is in a subdirectory)

    # 3 — archive, delete, restore
    archive = shutil.make_archive(root / "dst-backup", "zip", root_dir=dst)
    shutil.rmtree(dst)
    print(dst.exists())                  # False
    shutil.unpack_archive(archive, dst)
    print((dst / "sub" / "c.txt").exists())   # True
```

</details>